In [2]:

import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, MessagesState,START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
load_dotenv()
model =ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    streaming=True
)
def chatbot(state:MessagesState):
    response =model.invoke(state["messages"])
    return {"messages": [response]}
builder = StateGraph(MessagesState)
builder.add_node("chatbot",chatbot)
builder.add_edge(START,"chatbot")
builder.add_edge("chatbot", END)
checkpointer = InMemorySaver()
app = builder.compile(checkpointer=checkpointer)
config ={
    "configurable":{"thread_id":"chat-1"}
}
# app.invoke(
#     {"messages":[HumanMessage(content="My name is mayank")]},
#     config
# )
# result = app.invoke(
#    {"messages":[HumanMessage(content="what is my name?")]},
#     config
# )
# print(result["messages"][-1].content)

while True:
    user_input = input("\nYou:")
    if user_input.lower() in ["exit","quit"]:
        print("Chat ended!")
        break
#It will not waitining in the buffer
    print("AI: ",end="", flush=True)
    for message, metadata in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
        stream_mode="messages"
    ):
        print(message.content,end="", flush=True)
    
    print()

   

AI: I don’t actually know your name—unless you’ve told me before in this conversation, I don’t have that information. If you’d like me to address you by name, just let me know what it is!
AI: I’m not able to retain or look back at earlier parts of our conversation, so I can’t see the questions you asked before. If you’d like to revisit a particular topic, just let me know what it was and I’ll be happy to help!
Chat ended!
